<a href="https://colab.research.google.com/github/Dyl777/gaussian-splatting-colab/blob/main/gaussian_splatting_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import torch

if torch.cuda.is_available():
    print("CUDA is available! GPU Name:", torch.cuda.get_device_name(0))
else:
    print("CUDA is not available. Please ensure you have a GPU runtime selected (e.g., T4 GPU).")

CUDA is available! GPU Name: Tesla T4


## Step 1: Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATASET_PATH = '/content/drive/MyDrive/warehouse_dataset/warehouse_pics'
print('Files found:', os.listdir(DATASET_PATH))

Mounted at /content/drive
Files found: ['VID_20260427_113637.mp4', 'VID_20260427_115925.mp4', 'VID_20260427_121510.mp4', 'VID_20260427_145207.mp4', 'VID_20260529_132950.mp4', 'IMG_20260608_170409_628.jpg', 'IMG_20260608_170402_291.jpg', 'IMG_20260608_170401_380.jpg', 'IMG_20260608_170359_305.jpg', 'IMG_20260608_170356_085.jpg', 'IMG_20260608_170355_379.jpg', 'IMG_20260608_170350_191.jpg', 'IMG_20260608_170345_744.jpg', 'IMG_20260608_170344_296.jpg', 'IMG_20260608_170343_649.jpg', 'IMG_20260608_170342_224.jpg', 'IMG_20260608_170339_581.jpg', 'IMG_20260608_170338_169.jpg', 'IMG_20260608_170335_185.jpg', 'IMG_20260608_170334_779.jpg', 'VID_20260608_114854.mp4', 'IMG_20260608_114800_673.jpg', 'IMG_20260608_114758_374.jpg', 'IMG_20260608_114756_776.jpg', 'IMG_20260608_114749_196.jpg', 'IMG_20260608_114746_473.jpg', 'IMG_20260608_114744_266.jpg', 'IMG_20260608_114735_726.jpg', 'IMG_20260608_114729_892.jpg', 'IMG_20260608_114728_547.jpg', 'IMG_20260608_114726_602.jpg', 'IMG_20260608_114723_66

## Step 2: Install dependencies

In [35]:
%cd /content
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting
!pip install -q plyfile

# Install ninja for better compilation of CUDA extensions
!pip install ninja

%cd /content/gaussian-splatting
# Update all submodules to ensure latest source, especially for simple-knn
!git submodule update --init --recursive

# Force reinstall and show output to diagnose potential CUDA compilation issues for diff-gaussian-rasterization
!pip install --force-reinstall -v ./submodules/diff-gaussian-rasterization

# Check nvcc version for debugging CUDA compilation issues
print("Checking NVCC (CUDA Compiler) version:")
!nvcc --version

# Navigate into simple-knn directory and install from there for robustness
%cd /content/gaussian-splatting/submodules/simple-knn
!pip install --force-reinstall -v .

%cd /content/gaussian-splatting
!apt-get install -y colmap ffmpeg

/content
fatal: destination path 'gaussian-splatting' already exists and is not an empty directory.
/content/gaussian-splatting
Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
Processing ./submodules/diff-gaussian-rasterization
  Running command python setup.py egg_info
  running egg_info
  creating /tmp/pip-pip-egg-info-h03ex8x8/diff_gaussian_rasterization.egg-info
  writing /tmp/pip-pip-egg-info-h03ex8x8/diff_gaussian_rasterization.egg-info/PKG-INFO
  writing dependency_links to /tmp/pip-pip-egg-info-h03ex8x8/diff_gaussian_rasterization.egg-info/dependency_links.txt
  writing top-level names to /tmp/pip-pip-egg-info-h03ex8x8/diff_gaussian_rasterization.egg-info/top_level.txt
  writing manifest file '/tmp/pip-pip-egg-info-h03ex8x8/diff_gaussian_rasterization.egg-info/SOURCES.txt'
  reading manifest file '/tmp/pip-pip-egg-info-h03ex8x8/diff_gaussian_rasterization.egg-info/SOURCES.txt'
  writing manifest file '/tmp/pip-pip-egg-info-h03ex8x8/diff_gaussian_

In [ ]:
import os

# Path to the simple_knn.cu file
simple_knn_cu_path = '/content/gaussian-splatting/submodules/simple-knn/simple_knn.cu'

# Read the original content of the file
with open(simple_knn_cu_path, 'r') as f:
    content = f.read()

# Check if #include <cfloat> is already present to avoid duplicates
if '#include <cfloat>' not in content:
    # Insert #include <cfloat> after #include <cuda_runtime_api.h>
    # This should resolve the FLT_MAX undefined error
    modified_content = content.replace(
        '#include <cuda_runtime_api.h>',
        '#include <cuda_runtime_api.h>\n#include <cfloat>'
    )

    # Write the modified content back to the file
    with open(simple_knn_cu_path, 'w') as f:
        f.write(modified_content)

    print(f'Successfully added #include <cfloat> to {simple_knn_cu_path}')
else:
    print(f'#include <cfloat> already present in {simple_knn_cu_path}')

Now that the `simple_knn.cu` file has been modified, I will re-attempt to install the `simple_knn` module. After this, we can try running COLMAP again.

In [37]:
%cd /content/gaussian-splatting/submodules/simple-knn
!pip install --force-reinstall -v .

/content/gaussian-splatting/submodules/simple-knn
Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
Processing /content/gaussian-splatting/submodules/simple-knn
  Running command python setup.py egg_info
  running egg_info
  creating /tmp/pip-pip-egg-info-7ql17xqa/simple_knn.egg-info
  writing /tmp/pip-pip-egg-info-7ql17xqa/simple_knn.egg-info/PKG-INFO
  writing dependency_links to /tmp/pip-pip-egg-info-7ql17xqa/simple_knn.egg-info/dependency_links.txt
  writing top-level names to /tmp/pip-pip-egg-info-7ql17xqa/simple_knn.egg-info/top_level.txt
  writing manifest file '/tmp/pip-pip-egg-info-7ql17xqa/simple_knn.egg-info/SOURCES.txt'
  reading manifest file '/tmp/pip-pip-egg-info-7ql17xqa/simple_knn.egg-info/SOURCES.txt'
  writing manifest file '/tmp/pip-pip-egg-info-7ql17xqa/simple_knn.egg-info/SOURCES.txt'
  Preparing metadata (setup.py) ... done
  Running command python setup.py bdist_wheel
  running bdist_wheel
  running build
  running build_ext
  build

## Step 3: Extract frames from any videos, copy images into one folder

In [20]:
import os, shutil, glob

IMAGES_DIR = '/content/my_scene/images'
os.makedirs(IMAGES_DIR, exist_ok=True)

VIDEO_EXTS = {'.mp4', '.mov', '.avi', '.mkv', '.webm'}
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}

files = os.listdir(DATASET_PATH)
frame_counter = 0

for f in files:
    src = os.path.join(DATASET_PATH, f)
    ext = os.path.splitext(f)[1].lower()

    if ext in VIDEO_EXTS:
        print(f'Extracting frames from video: {f}')
        out_pattern = os.path.join(IMAGES_DIR, f'vid_{frame_counter:04d}_%04d.png')
        # Extract 2 frames per second — increase fps= for more detail
        os.system(f'ffmpeg -i "{src}" -vf fps=2 "{out_pattern}" -hide_banner -loglevel error')
        frame_counter += 1

    elif ext in IMAGE_EXTS:
        shutil.copy(src, IMAGES_DIR)

total = len(os.listdir(IMAGES_DIR))
print(f'Total images ready for COLMAP: {total}')

Extracting frames from video: VID_20260427_113637.mp4
Extracting frames from video: VID_20260427_115925.mp4
Extracting frames from video: VID_20260427_121510.mp4
Extracting frames from video: VID_20260427_145207.mp4
Extracting frames from video: VID_20260529_132950.mp4
Extracting frames from video: VID_20260608_114854.mp4
Total images ready for COLMAP: 890


In [38]:
# Install xvfb and its dependencies for headless OpenGL context
!apt-get update
!apt-get install -y xvfb freeglut3-dev libgl1-mesa-dev libglu1-mesa-dev

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

With `xvfb` installed, I will now modify the COLMAP commands to run within a virtual framebuffer, which should resolve the OpenGL context issue and allow COLMAP to use the T4 GPU for SIFT features.

In [30]:
# Create a backup directory in your Drive
backup_dest = '/content/drive/MyDrive/colmap_backup_my_scene'
os.makedirs(backup_dest, exist_ok=True)

# Copy the files
print("Backing up your files to Google Drive... This might take a moment.")
shutil.copytree('/content/my_scene', backup_dest, dirs_exist_ok=True)
print(f"Backup complete! Your files are safe in Google Drive at: {backup_dest}")

Backing up your files to Google Drive... This might take a moment.
Backup complete! Your files are safe in Google Drive at: /content/drive/MyDrive/colmap_backup_my_scene


In [39]:
import os

# ==============================================================================
# 1. SETUP ENVIRONMENT & FORCE GPU INITIALIZATION
# ==============================================================================
SCENE_DIR = '/content/my_scene'
DB_PATH   = f'{SCENE_DIR}/colmap.db'

os.makedirs(f'{SCENE_DIR}/sparse', exist_ok=True)

# Force Linux to expose the T4 GPU hardware and CUDA compute stacks to xvfb
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["NVIDIA_VISIBLE_DEVICES"] = "all"
os.environ["NVIDIA_DRIVER_CAPABILITIES"] = "compute,utility,graphics"

# Force-reload NVIDIA kernel module to ensure it's in exclusive compute mode
!nvidia-smi -c EXCLUSIVE_PROCESS || true

# ==============================================================================
# 2. COLMAP PIPELINE EXECUTION
# ==============================================================================

# Feature extraction (Kept at single_camera 0 to avoid dimension conflict crashes)
!xvfb-run --server-args="-screen 0 1024x768x24" colmap feature_extractor \
    --database_path {DB_PATH} \
    --image_path {IMAGES_DIR} \
    --ImageReader.single_camera 0

# Feature matching (Forced onto GPU via environment configurations)
!xvfb-run --server-args="-screen 0 1024x768x24" colmap exhaustive_matcher \
    --database_path {DB_PATH}

# Sparse reconstruction
!xvfb-run --server-args="-screen 0 1024x768x24" colmap mapper \
    --database_path {DB_PATH} \
    --image_path {IMAGES_DIR} \
    --output_path {SCENE_DIR}/sparse

print('COLMAP done. Sparse models:', os.listdir(f'{SCENE_DIR}/sparse'))

Compute mode is already set to EXCLUSIVE_PROCESS for GPU 00000000:00:04.0.
All done.
/usr/bin/xvfb-run: 184: colmap: not found
/usr/bin/xvfb-run: 184: colmap: not found
/usr/bin/xvfb-run: 184: colmap: not found
COLMAP done. Sparse models: []


In [ ]:
# 1. Properly purge old packages separately
!apt-get remove -y colmap
!apt-get autoremove -y

# 2. Add libopencv-dev to the dependency tree
!apt-get update
!apt-get install -y git cmake ninja-build build-essential libboost-program-options-dev libboost-graph-dev libboost-system-dev libeigen3-dev libfreeimage-dev libmetis-dev libgoogle-glog-dev libgflags-dev libglew-dev libcgal-dev libsqlite3-dev nvidia-cuda-toolkit libceres-dev libfaiss-dev libopenimageio-dev openimageio-tools libopencv-dev

# 3. Clear out the old directory, re-clone fresh, configure, compile, and install in a single chained path
!rm -rf colmap
!git clone https://github.com/colmap/colmap

# Chained sequence to ensure directory context (cd) isn't lost between exclamation points
!cd colmap && mkdir -p build && cd build && echo "==============================================================================" && echo "⚡ RUNNING CMAKE CONFIGURATION" && echo "==============================================================================" && cmake .. -GNinja -DCUDA_ENABLED=ON -DGUI_ENABLED=OFF -DOPENGL_ENABLED=OFF -DCOLMAP_FIND_QUIETLY=OFF && echo "==============================================================================" && echo "⚡ RUNNING NINJA COMPILATION (LIVE TERMINAL STREAM)" && echo "==============================================================================" && ninja -j$(nproc) && echo "==============================================================================" && echo "⚡ INSTALLING BINARIES" && echo "==============================================================================" && ninja install

# 4. VERIFYING COLMAP COMPILATION MODE
!echo "=============================================================================="
!echo "⚡ VERIFYING COLMAP COMPILATION MODE"
!echo "=============================================================================="
!colmap help | head -n 3

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Package 'colmap' is not installed, so not removed
0 upgraded, 0 newly installed, 0 to remove and 127 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
0 upgraded, 0 newly installed, 0 to remove and 127 not upgraded.
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hi

In [ ]:
import os
import shutil

# ==============================================================================
# 1. SECURE BACKUP TO GOOGLE DRIVE (Saves your existing progress)
# ==============================================================================
SCENE_DIR = '/content/my_scene'
DB_PATH   = f'{SCENE_DIR}/colmap.db'
IMAGES_DIR = '/content/my_scene/images' # Double check if your image directory path matches

# ==============================================================================
# 2. HEADLESS ENVIRONMENT ALLOCATION (Fixes Issue #570 Context Failures)
# ==============================================================================
os.makedirs(f'{SCENE_DIR}/sparse', exist_ok=True)

# Explicit headless runtime setups matching Issue #570 configurations
os.environ["QT_QPA_PLATFORM"] = "offscreen"
os.environ["XDG_RUNTIME_DIR"] = "/tmp/runtime-root"
os.makedirs("/tmp/runtime-root", exist_ok=True)

# Define standard virtual framebuffer arguments for stable OpenGL virtualization
XVFB_CMD = 'xvfb-run --server-args="-screen 0 1024x768x24 -ac +extension GLX +render"'

# ==============================================================================
# 3. COLMAP PIPELINE EXECUTION
# ==============================================================================

print("--- Starting Feature Extraction ---")
# Feature extraction (Kept at single_camera 0 to bypass image dimension errors)
!{XVFB_CMD} colmap feature_extractor \
    --database_path {DB_PATH} \
    --image_path {IMAGES_DIR} \
    --ImageReader.single_camera 0


# processes gracefully in headless cloud setups.
!{XVFB_CMD} colmap exhaustive_matcher \
    --database_path {DB_PATH}

print("\n--- Starting Sparse Reconstruction Mapping ---")
# Sparse reconstruction mapper
!{XVFB_CMD} colmap mapper \
    --database_path {DB_PATH} \
    --image_path {IMAGES_DIR} \
    --output_path {SCENE_DIR}/sparse

print('\nCOLMAP complete. Sparse models found:', os.listdir(f'{SCENE_DIR}/sparse'))

## Step 5: Train Gaussian Splatting

In [ ]:
%cd /content/gaussian-splatting

# COLMAP mapper outputs to sparse/0, sparse/1, etc. — use the first model
!python train.py -s /content/my_scene --model_path /content/my_scene/output

## Step 6 (Optional): Render and export video

In [ ]:
!python render.py -m /content/my_scene/output

!ffmpeg -framerate 24 \
    -i /content/my_scene/output/train/ours_30000/renders/%05d.png \
    -vf "pad=ceil(iw/2)*2:ceil(ih/2)*2" \
    -c:v libx264 -r 24 -pix_fmt yuv420p \
    /content/warehouse_render.mp4 -y

print('Render saved to /content/warehouse_render.mp4')

## Step 7 (Optional): Copy output back to Google Drive

In [50]:
import shutil
OUTPUT_DRIVE = '/content/drive/MyDrive/warehouse_dataset/warehouse_output'
shutil.copytree('/content/my_scene/output', OUTPUT_DRIVE, dirs_exist_ok=True)
print('Output copied to Google Drive:', OUTPUT_DRIVE)

Output copied to Google Drive: /content/drive/MyDrive/warehouse_dataset/warehouse_output
